# Homologación arancelaria uno a uno, 2011–2025

Este cuaderno cruza la base depurada con la tabla de correspondencias arancelarias por año y código de subpartida. Para cada registro seleccionado, conserva el código original y añade la columna `subpartida_homologada`.

## Archivos de entrada

- `datos/procesados/exportaciones_2011_2025_base_depurada.parquet`, generado por el cuaderno 01.
- `datos/referencias/tabla_homologacion_final.parquet`, que contiene las correspondencias por año.

## Archivos de salida

- `datos/procesados/exportaciones_1a1_sin_agregacion.parquet`, base que utiliza el cuaderno 03 para calcular los indicadores.
- `resultados/homologacion/auditoria_registros_sin_correspondencia_anual.csv`, detalle de los registros cuyo código figura en la selección uno a uno, pero carece de correspondencia para su año de exportación.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# 1. Rutas de entrada y salida
PROCESADOS = Path("datos/procesados")
ARCHIVO_BASE = PROCESADOS / "exportaciones_2011_2025_base_depurada.parquet"
ARCHIVO_TABLA = Path("datos/referencias/tabla_homologacion_final.parquet")

SALIDA = Path("resultados/homologacion")
SALIDA.mkdir(parents=True, exist_ok=True)

In [3]:
# 2. Lectura de las bases y normalización de las claves de homologación.
base = pd.read_parquet(ARCHIVO_BASE)
tabla = pd.read_parquet(ARCHIVO_TABLA, columns=["Año", "codigo_original", "grupo", "serie"])

base["Año"] = pd.to_numeric(base["Año"], errors="raise").astype("int64")
tabla["Año"] = pd.to_numeric(tabla["Año"], errors="raise").astype("int64")

base["Subpartida"] = base["Subpartida"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(10)
tabla["codigo_original"] = tabla["codigo_original"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(10)
tabla["serie"] = tabla["serie"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(10)

# Validaciones de unicidad, formato y cantidad de registros.
assert not tabla.duplicated(["Año", "codigo_original"]).any()
assert base["Subpartida"].str.fullmatch(r"\d{10}").all()
assert len(base) == 1_988_937

In [4]:
# 3. Selección de correspondencias uno a uno y validación de continuidad.
uno_a_uno = tabla.loc[tabla["grupo"].eq("uno_a_uno")].copy()

assert uno_a_uno["serie"].nunique() == 5_626
assert uno_a_uno.groupby("serie")["Año"].nunique().eq(15).all()
assert not uno_a_uno.duplicated(["Año", "serie"]).any()

# Preparación de la tabla de correspondencias para el cruce con la base original.
reglas = uno_a_uno[["Año", "codigo_original", "serie"]].rename(
    columns={"codigo_original": "Subpartida", "serie": "subpartida_homologada"}
)

In [5]:
# 4. Homologación mediante cruce por año y subpartida arancelaria.
cruce = base[["Año", "Subpartida"]].merge(
    reglas, on=["Año", "Subpartida"], how="left", sort=False, validate="many_to_one"
)
assert len(cruce) == len(base)

# Incorporación de la subpartida homologada a los registros con correspondencia.
seleccionado = cruce["subpartida_homologada"].notna().to_numpy()
homologada = base.loc[seleccionado].copy()
homologada["subpartida_homologada"] = cruce.loc[
    seleccionado, "subpartida_homologada"
].to_numpy()

In [6]:
# 5. Validación de los registros homologados y sus valores FOB.
fob_final = homologada["US$ millones"].sum()

assert len(homologada) == 1_671_785
assert homologada["Subpartida"].nunique() == 5_804
assert homologada["subpartida_homologada"].nunique() == 5_626
assert np.isclose(fob_final, 706_350.99033953, rtol=0, atol=1e-5)

# Identificación de registros cuyo código existe en la tabla, pero no tiene correspondencia para el año de exportación.
codigos_candidatos = set(uno_a_uno["codigo_original"])
en_catalogo = base["Subpartida"].isin(codigos_candidatos).to_numpy()
fuera_de_anio = en_catalogo & ~seleccionado
excluidos = base.loc[
    fuera_de_anio,
    ["Año", "Mes", "Subpartida", "Descripción", "US$ millones", "TM"],
].copy()

# Validación de los registros excluidos y conciliación de sus valores FOB.
assert len(excluidos) == 450
assert excluidos["Subpartida"].nunique() == 66
assert np.isclose(excluidos["US$ millones"].sum(), 44.81721081, atol=1e-6, rtol=0)
assert np.isclose(
    base.loc[en_catalogo, "US$ millones"].sum() - fob_final,
    excluidos["US$ millones"].sum(),
    rtol=0, atol=1e-5,
)

# Resumen de los resultados de la homologación.
print("Registros homologados (sin agregación):", len(homologada))
print("Códigos originales distintos:", homologada["Subpartida"].nunique())
print("Subpartidas homologadas:", homologada["subpartida_homologada"].nunique())
print(f"FOB seleccionado (millones US$): {fob_final:,.6f}")
print("Registros candidatos sin regla en el año correspondiente:", len(excluidos))
print(f"FOB de dichos registros: {excluidos['US$ millones'].sum():,.6f}")

Registros homologados (sin agregación): 1671785
Códigos originales distintos: 5804
Subpartidas homologadas: 5626
FOB seleccionado (millones US$): 706,350.990340
Registros candidatos sin regla en el año correspondiente: 450
FOB de dichos registros: 44.817211


In [7]:
# 6. Guardado de la base homologada y del registro de auditoría
homologada.to_parquet(
    PROCESADOS / "exportaciones_1a1_sin_agregacion.parquet",
    index=False
)

excluidos.to_csv(
    SALIDA / "auditoria_registros_sin_correspondencia_anual.csv",
    index=False,
    encoding="utf-8-sig"
)